# Explore CodePulse DB
Walks through every table in `database/codepulse.db` so we can see what the miner produced.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# Notebook lives in notebooks/, DB lives in database/ — go up one level
DB_PATH = Path('..') / 'database' / 'codepulse.db'
assert DB_PATH.exists(), f'DB not found at {DB_PATH.resolve()}'

conn = sqlite3.connect(DB_PATH)
print(f'Connected to {DB_PATH.resolve()}')

## What tables exist?

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
tables

## How many rows in each?

In [ ]:
counts = {
    name: pd.read_sql(f'SELECT COUNT(*) AS n FROM {name}', conn).iloc[0, 0]
    for name in tables['name']
}
pd.Series(counts, name='rows').to_frame()

## events — one row per (commit, file) pair

In [ ]:
events = pd.read_sql('SELECT * FROM events LIMIT 5', conn)
print('columns:', list(events.columns))
events

## pull_requests

In [ ]:
pd.read_sql('SELECT * FROM pull_requests LIMIT 5', conn)

## issues

In [ ]:
pd.read_sql('SELECT * FROM issues LIMIT 5', conn)

## Link tables — commit ↔ PR and commit ↔ issue

In [ ]:
pd.read_sql('SELECT * FROM pr_commits LIMIT 5', conn)

In [ ]:
pd.read_sql('SELECT * FROM commit_issues LIMIT 5', conn)

## Quick sanity checks
Most-touched files and most-active authors — useful for the bug-prediction features.

In [ ]:
pd.read_sql(
    '''
    SELECT file_path, COUNT(*) AS touches
    FROM events
    GROUP BY file_path
    ORDER BY touches DESC
    LIMIT 10
    ''',
    conn,
)

In [ ]:
pd.read_sql(
    '''
    SELECT author_name, COUNT(DISTINCT commit_hash) AS commits
    FROM events
    GROUP BY author_name
    ORDER BY commits DESC
    LIMIT 10
    ''',
    conn,
)

In [ ]:
conn.close()